# Zombie-Brain v6 on Colab
Runtime → Change runtime type → **T4 GPU**. Upload `zombie_brain_v6.zip` and your dataset folder (or mount Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -15

In [ ]:
!unzip -q -o zombie_brain_v6.zip -d /content
%cd /content/zombie_brain_v6
!pip -q install tokenizers matplotlib

## 1. Smoke test (tiny models, a few minutes) — must print `SMOKE TEST PASSED`

In [ ]:
!python smoke_test.py 2>&1 | tail -25

## 2. Prepare data (your ~200 MB of text in `/content/mydata`)
Tokenizer is trained on the train split only; 10% of blocks held out as val; 5000 synthetic facts added to train for the recall test.

In [ ]:
# Option A: one pool of text files, random 10% held out for val
!python prep_data.py --input /content/mydata --out data --vocab 8192 --kv-pairs 5000

# Option B: you already have train.txt / validation.txt / test.txt (use this instead of A)
# !python prep_data.py --train /content/mydata/train.txt --val /content/mydata/validation.txt --test /content/mydata/test.txt --out data --vocab 8192 --kv-pairs 5000

## 3. Time one config (50 steps) before committing hours
Read the `tok/s` lines. Zombie trains slower per step than the baselines (it runs both paths + memory bookkeeping).

In [ ]:
!python train.py --name gqa1x  --steps 50 --out /content/tmp --eval-every 0 --no-ptq --force
!python train.py --name zombie --steps 50 --out /content/tmp --eval-every 0 --no-ptq --force

## 4. Sweep (1 seed, all 17 configs) → then 3 seeds on the headline five
Runs go to Drive; if the session dies, re-run the same cell (finished runs skip, partial runs resume).

In [ ]:
OUT = '/content/drive/MyDrive/zombie_runs'
!python run_ablations.py --suite sweep --out $OUT

In [ ]:
!python run_ablations.py --suite headline --seeds 3 --out $OUT

## 5. Results

In [ ]:
!python aggregate.py --runs $OUT
from IPython.display import Image
Image(OUT + '/pareto.png')

## 6. CPU speed
Do this on your own machine (the i3): `python bench_cpu.py runs/gqa1x_s0 runs/zombie_s0 --threads 6` using `model.pt` + `config.json` from each run folder.